# IOAI — 2025 Regional Caloric Consumption (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, zipfile, urllib.request
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/train.csv'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/roai-caloric.zip', '/tmp/cal.zip')
    with zipfile.ZipFile('/tmp/cal.zip') as zf: zf.extractall('data')
print('데이터 준비:', 'data/train.csv' if os.path.exists('data/train.csv') else '실패')
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Caloric Consumption — 모범답안 (Gender 인코딩 + 다항특징 + Ridge)
원대회 reference: `Gender` 더미 인코딩 + `PolynomialFeatures(3)` + `Ridge` 로 비선형 상호작용까지 포착.
채점 = held-out(`User_ID%5==0`) **MAE**(낮을수록 좋음).

In [ ]:
import pandas as pd, numpy as np
from sklearn.linear_model import Ridge
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import mean_absolute_error

df = pd.read_csv('data/train.csv')
is_val = df['User_ID'] % 5 == 0
tr, va = df[~is_val].reset_index(drop=True), df[is_val].reset_index(drop=True)

In [ ]:
def prep(d):
    g = (d['Gender']=='male').astype(int).rename('Gender_male')
    return pd.concat([d[['Age','Height','Weight','Duration','Heart_Rate','Body_Temp']], g], axis=1)

poly = PolynomialFeatures(3)
Xtr = poly.fit_transform(prep(tr)); Xva = poly.transform(prep(va))
m = Ridge(alpha=10).fit(Xtr, tr['Calories'])
pred = m.predict(Xva)
print('held-out MAE:', round(mean_absolute_error(va['Calories'], pred), 4))
pd.DataFrame({'User_ID': va['User_ID'], 'Calories': pred}).to_csv('submission.csv', index=False)
print('wrote submission.csv', len(va))

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)